In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

from options import OptionSurface, Deribit, OKX, Bybit
from portfolio_management import Portfolio
from api_client import TradingDeskAPI
from scanner import MarketScanner

In [2]:
# Initialize all classes and parameters

load_dotenv(r"D:/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
# load_dotenv(r"C:/Users/brian/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
BASE_URL = os.getenv("BASE_URL")
USER_EMAIL = os.getenv("USER_EMAIL")
USER_PASSWORD = os.getenv("USER_PASSWORD")
JWT_TOKEN = os.getenv("JWT")
DATA_DIR = os.getenv("DATA_DIR")
FRACTION = float(os.getenv("FRACTION"))
MAX_POSITION = float(os.getenv("MAX_POSITION"))
ENTRY_EV_THRESHOLD = float(os.getenv("ENTRY_EV_THRESHOLD")) # require 1% edge, default = 0
EXIT_EV_THRESHOLD = float(os.getenv("EXIT_EV_THRESHOLD"))
print(BASE_URL)

with open(f"{DATA_DIR}/crypto_tag_ids.json", "r") as f:
    crypto_tag_ids = set(json.load(f))

s = OptionSurface()
portfolio = Portfolio()
api = TradingDeskAPI(base_url=BASE_URL, email=USER_EMAIL, password=USER_PASSWORD, token=JWT_TOKEN)
scanner = MarketScanner(api=api)

currencies = ["BTC", "ETH"]

https://alphasignal-dev.moretoncp.com


In [3]:
# 1. Get all dfs
orders_df = pd.read_parquet(f"{DATA_DIR}/orders.parquet")
fills_df = pd.read_parquet(f"{DATA_DIR}/fills.parquet")
positions_df = pd.read_parquet(f"{DATA_DIR}/positions.parquet")
equity_df = pd.read_parquet(f"{DATA_DIR}/equity.parquet")

# fills_df = fills_df.iloc[0:0]

In [4]:
# 2. Get newly executed trades
fills_df = portfolio.sync_fills(api=api, fills_df=fills_df)
fills_df

,question,order_id,condition_id,token_id,outcome,side,price,shares,fee,timestamp,fill_id
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,BUY,0.981,4.87,0.00635,2026-08-14 08:25:11+00:00,4c3c68e2-7b89-463c-ae03-48b92808eeef
1,"Will Ethereum reach $4,500 by December 31, 2026?",0x387e06d4de5bb3110d135297db63ce667c4a3ac0de7d...,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,BUY,0.960,4.95,0.01330,2026-08-19 08:00:45+00:00,ff80c017-a13d-43ee-85c0-35a2c289c1b6
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xf7ff5b16da1bb114dc0b34d7e83324beb8b85502f029...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,BUY,0.969,4.91,0.01032,2026-08-19 08:01:06+00:00,cd0ebe33-7e52-44c6-a022-466914d90934
3,"Will Ethereum reach $5,500 by December 31, 2026?",0x007e608ee1b924c6137bc7fe08a909950bc98b3cf9e7...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,BUY,0.031,40.00,0.08410,2026-08-29 03:53:04+00:00,9c890c35-132a-44b6-ba48-b2fe16061fd3
4,"Will Ethereum reach $6,000 by December 31, 2026?",0x439db8be33af7e7d3497392a9f722930b7285e696ea4...,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,BUY,0.960,40.00,0.10752,2026-08-29 03:54:04+00:00,dbcbca55-34bd-446b-8a76-02c65229d62f
5,"Will Ethereum reach $6,000 by December 31, 2026?",0x6aba1d2deb08b681a8db29c758293485c023f541cb55...,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,BUY,0.960,46.00,0.12364,2026-08-29 04:02:15+00:00,0f6bafb7-aab6-49bf-a864-eb7be2c5ab5f
6,"Will Ethereum reach $5,500 by December 31, 2026?",0x1bb46bc40417ee40111bd4f3ef64b903fdecec03f2c2...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,BUY,0.031,35.00,0.07359,2026-08-29 04:03:17+00:00,567da504-7b00-4795-8543-c02e0bce9e0b
7,"Will Ethereum reach $5,500 by December 31, 2026?",0x1bb46bc40417ee40111bd4f3ef64b903fdecec03f2c2...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,BUY,0.031,11.00,0.00000,2026-08-29 09:06:59+00:00,69825e95-8228-4413-ac3b-60c9da86ec17


In [5]:
# 3. Reconstruct portfolio
# 4. Calculate realized P&L
positions_df = portfolio.reconstruct_positions_fifo(fills_df=fills_df)
positions_df

,question,condition_id,token_id,outcome,shares,cost_basis,avg_entry_price,realized_pnl,realized_shares,realized_fees
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,4.87,4.78382,0.982304,0.0,0.0,0.0
1,"Will Ethereum reach $4,500 by December 31, 2026?",0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,4.95,4.76530,0.962687,0.0,0.0,0.0
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,4.91,4.76811,0.971102,0.0,0.0,0.0
3,"Will Ethereum reach $5,500 by December 31, 2026?",0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,86.00,2.82369,0.032834,0.0,0.0,0.0
4,"Will Ethereum reach $6,000 by December 31, 2026?",0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,86.00,82.79116,0.962688,0.0,0.0,0.0


In [6]:
# 5. Sync positions with api
portfolio.reconcile_positions(api=api, positions_df=positions_df)
positions_df

POSITION SYNCED: 96993471854400156408670527613150944443359272190785251193551242374636006072800
POSITION SYNCED: 4251240413067872674686064123803499349976238079777545616008767482027755117695
POSITION SYNCED: 447913121087537662187327157218300928168985707076581166939650812231975265268
POSITION SYNCED: 61710247276022470252804012285430368172565001541144070158233974925141072188846
POSITION SYNCED: 99625432243305856023409516897537718601994293111805554742206464299633923525276


,question,condition_id,token_id,outcome,shares,cost_basis,avg_entry_price,realized_pnl,realized_shares,realized_fees
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,4.87,4.78382,0.982304,0.0,0.0,0.0
1,"Will Ethereum reach $4,500 by December 31, 2026?",0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,4.95,4.76530,0.962687,0.0,0.0,0.0
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,4.91,4.76811,0.971102,0.0,0.0,0.0
3,"Will Ethereum reach $5,500 by December 31, 2026?",0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,86.00,2.82369,0.032834,0.0,0.0,0.0
4,"Will Ethereum reach $6,000 by December 31, 2026?",0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,86.00,82.79116,0.962688,0.0,0.0,0.0


In [7]:
# 6. Get latest order state
orders_df = portfolio.sync_orders(api=api, orders_df=orders_df, fills_df=fills_df)
orders_df

,question,order_id,condition_id,token_id,outcome,side,price,requested_size,order_type,status,created_at,cancelled_at,filled_size,remaining_size
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,BUY,0.981,4.87,GTC,FILLED,2026-08-14 09:28:01.978131+00:00,NaT,4.87,0.0
1,"Will Ethereum reach $4,500 by December 31, 2026?",0x387e06d4de5bb3110d135297db63ce667c4a3ac0de7d...,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,BUY,0.960,4.95,GTC,FILLED,2026-08-19 08:00:40.748876+00:00,NaT,4.95,0.0
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xf7ff5b16da1bb114dc0b34d7e83324beb8b85502f029...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,BUY,0.969,4.91,GTC,FILLED,2026-08-19 08:01:02.498064+00:00,NaT,4.91,0.0
3,"Will Ethereum reach $5,500 by December 31, 2026?",0x007e608ee1b924c6137bc7fe08a909950bc98b3cf9e7...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,BUY,0.031,40.00,GTC,FILLED,2026-08-29 03:53:04+00:00,NaT,40.00,0.0
4,"Will Ethereum reach $6,000 by December 31, 2026?",0x439db8be33af7e7d3497392a9f722930b7285e696ea4...,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,BUY,0.960,40.00,GTC,FILLED,2026-08-29 03:54:04+00:00,NaT,40.00,0.0
5,"Will Ethereum reach $6,000 by December 31, 2026?",0x6aba1d2deb08b681a8db29c758293485c023f541cb55...,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,BUY,0.960,46.00,GTC,FILLED,2026-08-29 04:02:15+00:00,NaT,46.00,0.0
6,"Will Ethereum reach $5,500 by December 31, 2026?",0x1bb46bc40417ee40111bd4f3ef64b903fdecec03f2c2...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,BUY,0.031,46.00,GTC,FILLED,2026-08-29 04:03:17+00:00,NaT,46.00,0.0


In [8]:
# 7. Mark positions to market
positions_df = portfolio.mark_positions_to_market(api=api, positions_df=positions_df)
positions_df

,question,condition_id,token_id,outcome,shares,cost_basis,avg_entry_price,realized_pnl,realized_shares,realized_fees,current_price,fee_rate,market_value,market_value_after_fees,unrealized_pnl,unrealized_pnl_after_fees,unrealized_return
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,4.87,4.78382,0.982304,0.0,0.0,0.0,0.985,0.07,4.79695,4.791913,0.01313,0.008093,0.002745
1,"Will Ethereum reach $4,500 by December 31, 2026?",0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,4.95,4.76530,0.962687,0.0,0.0,0.0,0.900,0.07,4.45500,4.423815,-0.31030,-0.341485,-0.065117
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,4.91,4.76811,0.971102,0.0,0.0,0.0,0.946,0.07,4.64486,4.627302,-0.12325,-0.140808,-0.025849
3,"Will Ethereum reach $5,500 by December 31, 2026?",0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,86.00,2.82369,0.032834,0.0,0.0,0.0,0.050,0.07,4.30000,4.014050,1.47631,1.190360,0.522830
4,"Will Ethereum reach $6,000 by December 31, 2026?",0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,86.00,82.79116,0.962688,0.0,0.0,0.0,0.949,0.07,81.61400,81.322638,-1.17716,-1.468522,-0.014218


In [14]:
# 8. Calculate equity
equity_df = portfolio.calculate_equity(api=api, positions_df=positions_df, equity_df=equity_df)
equity_df

Equity:  99.97
Return:  0.7953%
Sharpe:  -0.0003
Sortino: -0.0004
Max Drawdown: -1.33%


,timestamp,cash,market_value,equity,realized_pnl,unrealized_pnl,period_return,sharpe,sortino,max_drawdown
0,2026-08-14 07:47:24.047885+00:00,100.00000,0.00000,100.00000,0.0,0.00000,NaN,NaN,NaN,NaN
1,2026-08-15 14:14:05.272655+00:00,95.21618,4.79695,100.01313,0.0,0.01948,0.000131,NaN,NaN,NaN
2,2026-08-17 02:58:57.343900+00:00,95.21618,4.77747,99.99365,0.0,0.00000,-0.000195,NaN,NaN,NaN
3,2026-08-19 07:56:33.538278+00:00,95.21618,4.78234,99.99852,0.0,0.00487,0.000049,NaN,NaN,NaN
4,2026-08-19 08:05:37.263592+00:00,85.68277,14.23772,99.92049,0.0,-0.04954,-0.000780,NaN,NaN,NaN
5,2026-08-24 13:32:14.011333+00:00,85.68747,13.80372,99.49119,0.0,-0.51351,-0.004296,NaN,NaN,NaN
6,2026-08-25 03:29:31.510439+00:00,85.68857,13.71458,99.40315,0.0,-0.60265,-0.000885,NaN,NaN,NaN
7,2026-08-27 02:38:40.684140+00:00,85.69107,13.69973,99.39080,0.0,-0.61750,-0.000124,NaN,NaN,NaN
8,2026-08-28 08:23:58.626786+00:00,85.69227,13.75861,99.45088,0.0,-0.55862,0.000604,NaN,NaN,NaN
9,2026-08-29 06:23:37.062020+00:00,0.41962,98.38844,98.80806,0.0,-1.20264,-0.006464,-0.553123,-0.414451,NaN


In [15]:
# 9. Get current markets
all_markets_df = api.get_all_markets(DATA_DIR=DATA_DIR, count_limit=10, liquidity_num_min=10000, volume_num_min=5000)
all_markets_df.head()

Fetched 100 markets | Total: 100
Fetched 100 markets | Total: 200
Fetched 100 markets | Total: 300
Fetched 100 markets | Total: 400
Fetched 100 markets | Total: 500
Fetched 100 markets | Total: 600
Fetched 100 markets | Total: 700
Fetched 100 markets | Total: 800
Fetched 100 markets | Total: 900
Fetched 100 markets | Total: 1,000


,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,version,negRiskMarketID,seriesColor,showGmpSeries,showGmpOutcome,oneHourPriceChange,umaResolutionStatus,eventStartTime,gameStartTime,groupItemRange
0,559651,Xi Jinping out before 2027?,0xa467b14d51f01b957109d9cbb1d6c124fab2a089d52e...,xi-jinping-out-before-2027,,2027-01-01T04:59:00Z,215957.991,2025-07-03T20:37:00.228Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN
1,559652,Will Gavin Newsom win the 2028 Democratic pres...,0x0f49db97f71c68b1e42a6d16e3de93d85dbf7d4148e3...,will-gavin-newsom-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,354493.02604,2025-07-11T18:35:56.805Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
2,559653,Will Alexandria Ocasio-Cortez win the 2028 Dem...,0xe6bcc2f1dd025ce5e1833190f7c60a71171c94f805df...,will-alexandria-ocasio-cortez-win-the-2028-dem...,,2028-11-07T00:00:00Z,341013.21586,2025-07-11T18:35:59.075Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,-0.0005,NaN,NaN,NaN,NaN
3,559654,Will Pete Buttigieg win the 2028 Democratic pr...,0x4c325469d9b516ef4e6b8f73a81a12607dec075e3c2f...,will-pete-buttigieg-win-the-2028-democratic-pr...,,2028-11-07T00:00:00Z,368348.88612,2025-07-11T18:35:58.818Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
4,559655,Will Josh Shapiro win the 2028 Democratic pres...,0xd65891729ce093cc12236856837eba1a0872fc7998fd...,will-josh-shapiro-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,339041.36154,2025-07-11T18:36:01.098Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN


In [16]:
# 10. Initialize variance surface
deribit = Deribit(currencies=currencies)
okx = OKX(currencies=currencies)
bybit = Bybit(currencies=currencies)

s.initialize(currencies=currencies, exchanges=[deribit, okx, bybit])

currency: BTC, spot: 79284.0, volume24h: 3.7369
currency: ETH, spot: 2490.4, volume24h: 5.2628
currency: BTC, spot: 79444.6, volume24h: 3686.60046071
currency: ETH, spot: 2492.4, volume24h: 80298.43347
currency: BTC, spot: 79439.3, volume24h: 4099.223858
currency: ETH, spot: 2492.34, volume24h: 62727.52419


In [17]:
# 11. Scan markets
markets_df, opportunities_df, arb_candidates_df = scanner.scan_market(markets_df=all_markets_df, s=s, crypto_tag_ids=crypto_tag_ids)
opportunities_df.head()

question: Will Bitcoin hit $150k by December 31, 2026?
event type: touch
direction: up
currency: BTC
required strike: 150000.0
iv: 0.5057855555573937
buy_yes_ev: -0.009854912536193243
sell_yes_ev: 0.005375982536193247
buy_no_ev: 0.005375982536193247
sell_no_ev: -0.00985491253619324
buy_yes_kelly: -0.00506821415808046
sell_yes_kelly: 0.1153953858050603
buy_no_kelly: 0.1153953858050603
sell_no_kelly: -0.005068214158080457

question: Will Bitcoin reach $200,000 by December 31, 2026?
event type: touch
direction: up
currency: BTC
required strike: 200000.0
iv: 0.5919882605446732
buy_yes_ev: -0.01270266065766354
sell_yes_ev: 0.007838490657663538
buy_no_ev: 0.00783849065766351
sell_no_ev: -0.012702660657663567
buy_yes_kelly: -0.006454828665359307
sell_yes_kelly: 0.35086994263530513
buy_no_kelly: 0.3508699426353048
sell_no_kelly: -0.006454828665359322

question: Will Bitcoin reach $190,000 by December 31, 2026?
event type: touch
direction: up
currency: BTC
required strike: 190000.0
iv: 0.586444

,question,yes_ask,yes_bid,no_ask,no_bid,model_prob,id,conditionId,slug,resolutionSource,...,buy_yes_ev,sell_yes_ev,buy_no_ev,sell_no_ev,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly,best_ev,best_action
787,"Will Ethereum dip to $1,500 by December 31, 2026?",0.122,0.119,0.881,0.878,0.190694,701552,0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...,will-ethereum-dip-to-1500-by-december-31-2026-...,,...,0.061196,-0.079033,-0.079033,0.061196,0.035150,-0.353896,-0.353896,0.035150,0.061196,buy_yes_ev
785,"Will Ethereum reach $4,000 by December 31, 2026?",0.18,0.15,0.85,0.82,0.114130,701548,0x9775cd557a3cdba4f0478070afa399c0805761dd9aa0...,will-ethereum-reach-4000-by-december-31-2026,,...,-0.076202,0.026945,0.026945,-0.076202,-0.047058,0.095500,0.095500,-0.047058,0.026945,buy_no_ev
783,"Will Ethereum reach $5,000 by December 31, 2026?",0.071,0.065,0.935,0.929,0.035530,701546,0x1c4fd67ab2a67f508672a69153559911244048b79a40...,will-ethereum-reach-5000-by-december-31-2026,,...,-0.040087,0.025216,0.025216,-0.040087,-0.021683,0.207553,0.207553,-0.021683,0.025216,sell_yes_ev
782,"Will Ethereum reach $5,500 by December 31, 2026?",0.054,0.05,0.95,0.946,0.021758,701545,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,will-ethereum-reach-5500-by-december-31-2026,,...,-0.035818,0.024917,0.024917,-0.035818,-0.019003,0.266923,0.266923,-0.019003,0.024917,buy_no_ev
780,"Will Ethereum reach $6,500 by December 31, 2026?",0.037,0.036,0.964,0.963,0.009033,701543,0x0f0499d1049385b1d53ffee6c42a1de7424e551c7652...,will-ethereum-reach-6500-by-december-31-2026,,...,-0.030461,0.024538,0.024538,-0.030461,-0.015857,0.365465,0.365465,-0.015857,0.024538,buy_no_ev


In [ ]:
# 773	Will Ethereum dip to $1,500 by December 31, 2026?	0.123	0.115	0.885	0.877	0.186221	701552	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...		...	0.055670	-0.078346	-0.078346	0.055670	0.032015	-0.363129	-0.363129	0.032015	0.055670	buy_yes_ev

# 758	Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.15	0.144	0.856	0.85	0.217894	701552	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...	...	0.058969	-0.082522	-0.082522	0.058969	0.035056	-0.304799	-0.304799	0.035056	0.058969	buy_yes_ev

# 3	    Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.142	0.141	0.859	0.858	0.198734	701552	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...	...	0.048206	-0.066213	-0.066213	0.048206	0.028374	-0.249818	-0.249818	0.028374	0.048206	buy_yes_ev

# 760	Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.15	0.146	0.854	0.85	0.209409	701552	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...	...	0.050484	-0.072136	-0.072136	0.050484	0.030011	-0.262750	-0.262750	0.030011	0.050484	buy_yes_ev

# 815	Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.172	0.155	0.845	0.828	0.221717	701552	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...	...	0.039748	-0.075885	-0.075885	0.039748	0.024295	-0.260180	-0.260180	0.024295	0.039748	buy_yes_ev

#       Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.122	0.121	0.879	0.878	0.219853	701552	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...	...	0.090355	-0.106299	-0.106299	0.090355	0.051898	-0.468050	-0.468050	0.051898	0.090355	buy_yes_ev

# 06	Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.146	0.89	0.264426	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...		59292.16479	...	0.109698	-0.161279	-0.161279	0.109698	0.064889	-0.781790	-0.781790	0.064889	0.109698	buy_yes_ev

# 585	Will Ethereum dip to $1,500 by December 31, 2026?	2027-01-01T05:00:00Z	0.402	0.599	0.525600	0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...	will-ethereum-dip-to-1500-by-december-31-2026-...		80206.24887	...	0.106772	-0.141414	-0.141414	0.106772	0

In [18]:
# 11. Scan arbitrage
cross_market_arb_df, vertical_arb_df = scanner.scan_arbitrage(arb_candidates_df)
vertical_arb_df

                                              question yes_ask yes_bid no_ask  \
766  Will Bitcoin reach $150,000 by December 31, 2026?   0.035   0.028  0.972   
166       Will Bitcoin hit $150k by December 31, 2026?   0.026   0.025  0.975   

    no_bid  model_prob      id  \
766  0.965    0.017918  701491   
166  0.974    0.017918  573656   

                                           conditionId  \
766  0xa7b594ae07d5c1590fa86028fcc2f870599043723741...   
166  0x02deb9538f5c123373adaa4ee6217b01745f1662bc90...   

                                                  slug resolutionSource  ...  \
766  will-bitcoin-reach-150000-by-december-31-2026-...                   ...   
166          will-bitcoin-hit-150k-by-december-31-2026                   ...   

    buy_yes_ev sell_yes_ev buy_no_ev sell_no_ev buy_yes_kelly sell_yes_kelly  \
766  -0.019446    0.008177  0.008177  -0.019446     -0.010101       0.156680   
166  -0.009855    0.005376  0.005376  -0.009855     -0.005068       0.115395 

,currency,event_type,direction,expiry,lower_strike,lower_question,lower_outcome,lower_price,lower_size,lower_cost,...,higher_question,higher_outcome,higher_price,higher_size,higher_cost,cost_per_unit,guaranteed_profit_per_unit,max_size,total_cost,total_guaranteed_profit


In [19]:
cross_market_arb_df

,currency,event_type,direction,expiry,A_strike,A_question,A_outcome,A_price,A_size,A_cost,...,B_question,B_outcome,B_price,B_size,B_cost,cost_per_unit,guaranteed_profit_per_unit,max_size,total_cost,total_guaranteed_profit


In [20]:
# 12. manage cancel orders
orders_df = portfolio.manage_open_orders(api=api, orders_df=orders_df, markets_df=markets_df, 
                ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)
orders_df

,question,order_id,condition_id,token_id,outcome,side,price,requested_size,order_type,status,created_at,cancelled_at,filled_size,remaining_size
0,"Will Bitcoin reach $200,000 by December 31, 2026?",0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,BUY,0.981,4.87,GTC,FILLED,2026-08-14 09:28:01.978131+00:00,NaT,4.87,0.0
1,"Will Ethereum reach $4,500 by December 31, 2026?",0x387e06d4de5bb3110d135297db63ce667c4a3ac0de7d...,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,BUY,0.960,4.95,GTC,FILLED,2026-08-19 08:00:40.748876+00:00,NaT,4.95,0.0
2,"Will Ethereum reach $5,500 by December 31, 2026?",0xf7ff5b16da1bb114dc0b34d7e83324beb8b85502f029...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,BUY,0.969,4.91,GTC,FILLED,2026-08-19 08:01:02.498064+00:00,NaT,4.91,0.0
3,"Will Ethereum reach $5,500 by December 31, 2026?",0x007e608ee1b924c6137bc7fe08a909950bc98b3cf9e7...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,BUY,0.031,40.00,GTC,FILLED,2026-08-29 03:53:04+00:00,NaT,40.00,0.0
4,"Will Ethereum reach $6,000 by December 31, 2026?",0x439db8be33af7e7d3497392a9f722930b7285e696ea4...,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,BUY,0.960,40.00,GTC,FILLED,2026-08-29 03:54:04+00:00,NaT,40.00,0.0
5,"Will Ethereum reach $6,000 by December 31, 2026?",0x6aba1d2deb08b681a8db29c758293485c023f541cb55...,0x6132cc91f7a0ad656e890b9040c159f4057f4c44359b...,9962543224330585602340951689753771860199429311...,No,BUY,0.960,46.00,GTC,FILLED,2026-08-29 04:02:15+00:00,NaT,46.00,0.0
6,"Will Ethereum reach $5,500 by December 31, 2026?",0x1bb46bc40417ee40111bd4f3ef64b903fdecec03f2c2...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,6171024727602247025280401228543036817256500154...,Yes,BUY,0.031,46.00,GTC,FILLED,2026-08-29 04:03:17+00:00,NaT,46.00,0.0


In [21]:
# Exit arbitrage positions
orders_df1 = orders_df.copy()
orders_df1 = portfolio.close_arbitrage_positions(api=api, positions_df=positions_df, orders_df=orders_df1,
        question_1="Will Ethereum reach $5,500 by December 31, 2026?", outcome_1="Yes",
        question_2="Will Ethereum reach $6,000 by December 31, 2026?", outcome_2="No")

EXIT ARBITRAGE SIGNAL
Leg 1 Market              : Will Ethereum reach $5,500 by December 31, 2026?
Leg 1 Outcome             : Yes
Leg 1 Size                : 86.0
Leg 1 Avg Entry Price     : 0.03283360465116279
                          : 
Leg 2 Market              : Will Ethereum reach $6,000 by December 31, 2026?
Leg 2 Outcome             : No
Leg 2 Size                : 86.0
Leg 2 Avg Entry Price     : 0.9626879069767441
                          : 
Close All Positions       : False
Max Size                  : 71.54
Normalized Max Size       : 14.46
Remaining Size            : 14.459999999999994
Avg Realized PnL          : -0.003234441627906988
Executable Total Realized PnL : -0.046770025939535026
Expected Total Realized PnL : -0.278161980000001
                         VARIABLES IN REQUEST                         
Leg 1 tokenId             : 61710247276022470252804012285430368172565001541144070158233974925141072188846
Leg 1 orderPrice          : 0.05
Leg 1 orderSize           : 14

In [22]:
# 13. Enter arbitrage positions
orders_df = portfolio.run_arbitrage_opportunities(api=api, arbitrage_df=vertical_arb_df, orders_df=orders_df)
orders_df = portfolio.run_arbitrage_opportunities(api=api, arbitrage_df=cross_market_arb_df, orders_df=orders_df)

In [18]:
# 13. Risk management
orders_df = portfolio.run_risk_management(api=api, positions_df=positions_df, 
            markets_df=markets_df, orders_df=orders_df, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)

pos: question                     Will Bitcoin reach $200,000 by December 31, 2026?
condition_id                 0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...
token_id                     9699347185440015640867052761315094444335927219...
outcome                                                                     No
shares                                                                    4.87
cost_basis                                                             4.78382
avg_entry_price                                                       0.982304
realized_pnl                                                               0.0
realized_shares                                                            0.0
realized_fees                                                              0.0
current_price                                                            0.985
fee_rate                                                                  0.07
market_value                                   

In [ ]:
# 14. New opportunities
orders_df = portfolio.run_new_opportunities(api=api, opportunities_df=opportunities_df, orders_df=orders_df, 
            ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, MAX_POSITION=MAX_POSITION, FRACTION=FRACTION)

In [23]:
#     # 15. Safe dfs
portfolio.save_snapshots(DATA_DIR, markets_df, "markets")
portfolio.save_snapshots(DATA_DIR, opportunities_df, "opportunities")
portfolio.save(DATA_DIR, orders_df, "orders")
portfolio.save(DATA_DIR, fills_df, "fills")
portfolio.save(DATA_DIR, positions_df, "positions")
portfolio.save(DATA_DIR, equity_df, "equity")

save_snapshots df saved at: data/20260907/markets_20260907_193058.parquet
save_snapshots df saved at: data/20260907/opportunities_20260907_193058.parquet
save df saved at:  data/orders.parquet
save df saved at:  data/fills.parquet
save df saved at:  data/positions.parquet
save df saved at:  data/equity.parquet


In [ ]:
# Polymarket    Model (exchanges: deribit, bybit, okx)    Edge

# 80k Dec-26        22%          20%       +2%
# 90k Dec-26        14%          12%       +2%
# 100k Dec-26        9.5%         5.9%     +3.6%
# 110k Dec-26        7%           4%       +3%
# 120k Dec-26        5%           2%       +3%

In [ ]:
# Keep trading journal

# Your thesis.
# Why you think the market is mispriced.
# Position size.
# Exit criteria.
# What actually happened.

# 1. Is the options-derived probability actually predictive?

# 2. Are you accounting for crypto risk premia / risk-neutral vs physical probabilities?

# 3. Are your touch probabilities correctly calibrated?

# 4. Are fees and prediction-market spreads killing the apparent EV?

# 5. Does your exit rule actually improve realized P&L?

# 6. Are multiple contracts giving you the same underlying exposure?

In [ ]:
# Stage 2 — Semi-automated execution

# The bot does:

# find opportunities
# calculate size
# prepare orders

# You approve:

# BUY YES
# Market: BTC above $150k
# Price: 0.43
# Size: $500
# Expected edge: +8%

# Click "confirm".

# This is useful because prediction markets can have:

# ambiguous wording
# resolution risks
# sudden news events

In [ ]:
# Yes — potentially a lot, but only if you structure it correctly. If you're being graded on Sortino, the important thing is not simply “arbitrage = good.” It's whether the strategy improves downside-adjusted returns relative to the capital and risk it consumes.

# For your specific setup, I'd think about it this way:

# 1. Why the vertical arbitrage can help Sortino

# Your 75 paired contracts:

# YES $5,500
# NO $6,000

# have a very constrained payoff structure. Once both legs are filled at sufficiently favorable prices, the pair has approximately:

# $$ \$1-(P_{5500,YES}+P_{6000,NO}) $$

# of gross locked-in value per pair.

# That is attractive from a Sortino perspective because you're converting some capital into a low-variance / low-downside-return stream.

# If the pair costs $0.991 and ultimately pays $1:

# $$ \text{return} \approx \frac{0.009}{0.991}=0.91\% $$

# before fees.

# After your stated fees, it's roughly:

# $$ \frac{0.00421}{0.991}\approx\boxed{0.42\%} $$

# per completed pair.

# The key word is completed. If you only buy one side and fail to acquire the other, you haven't created the low-downside arbitrage payoff—you've created directional exposure.

# 2. But there's a subtle Sortino issue

# If your grading period is short, an arbitrage trade that makes $0.30 over several weeks may actually hurt your Sortino if it ties up capital that could have generated larger returns elsewhere.

# Sortino essentially asks:

# How much return am I generating per unit of downside volatility?

# So you care about:

# $$ \text{Sortino} = \frac{R_p-R_{\text{target}}} {\text{downside deviation}} $$

# A low-risk arbitrage can improve the denominator dramatically, but only if its return is meaningful relative to the capital allocated and the evaluation horizon.

# 3. Your residual NO position is different

# This is where your interview explanation becomes interesting.

# Your 11 excess NO $6,000 contracts aren't part of the arbitrage. They're directional.

# So conceptually your book is:

# Core

# 75 matched vertical arbitrage pairs → low downside / low variance

# Satellite

# 11 excess NO $6,000 → directional model exposure

# That's actually a reasonable portfolio construction approach.

# You're not saying:

# “I accidentally have an imbalance.”

# You're saying:

# “I deliberately allowed a residual directional position because my model's expected value for that exposure justified retaining it, while using the rest of my capital for a lower-risk relative-value strategy.”

# That is a much stronger trading argument.

# 4. However, don't claim the arbitrage automatically improves Sortino

# An interviewer could challenge you:

# “Why not just allocate all your capital to the arbitrage?”

# Your answer needs to be:

# “Because I was optimizing portfolio-level downside-adjusted return, not maximizing the number of arbitrage contracts. Once I accounted for fees, execution risk, capital utilization, and the expected return of the existing directional position, the marginal arbitrage wasn't necessarily the highest-Sortino use of capital.”

# That's a sophisticated answer.

# The biggest thing I'd watch

# Your stated edge is only about 0.42¢ per pair after fees.

# That's extremely thin.

# At that level, execution risk may dominate the theoretical arbitrage edge. A 1-minute delay, partial fill, or a few ticks of adverse movement can erase the expected profit.

# So for an interview, I'd explicitly say:

# “I only consider the vertical a true arbitrage after both legs are executable at prices that leave positive expected edge after fees and execution costs. Otherwise, I treat the unhedged leg as directional risk.”

# That's probably the strongest way to present it.

# Bottom line: Yes, the strategy can improve your Sortino because it can add positive, relatively low-downside P&L. But your portfolio construction and execution discipline are what make it Sortino-positive—not merely labeling the trade “arbitrage.”

In [ ]:
# Day 1 — API connection + market ingestion

# Goal:

# Can I pull markets automatically?

# Build:

# get_markets()

# Output:

# {
#  "condition_id": "...",
#  "question": "Will X happen?",
#  "tokens": [
#     {
#       "token_id": "YES",
#       "price": 0.42
#     },
#     {
#       "token_id": "NO",
#       "price": 0.58
#     }
#  ]
# }

# Store:

# markets

# condition_id
# question
# yes_token
# no_token
# created_time
# Day 2 — Historical snapshots

# Goal:

# Can I reconstruct what the market looked like yesterday?

# Every 5 minutes:

# while True:

#     markets = api.get_markets()

#     for m in markets:
#         database.save_snapshot(m)

#     sleep(300)

# Database:

# market_snapshots

# timestamp
# condition_id
# yes_price
# no_price
# volume
# Day 3 — Order book collection

# Now collect microstructure data.

# For selected markets:

# get_orderbook(token_id)

# Store:

# orderbook_snapshots

# timestamp

# token_id

# best_bid
# best_ask

# bid_depth
# ask_depth

# Calculate:

# Spread
# spread=ask−bid

# Example:

# Bid:
# 0.42

# Ask:
# 0.46

# Spread:
# 4 cents
# Day 4 — Build your scanner

# Your first scanner should be dumb but useful.

# Signal 1: Large moves
# if abs(price_change_24h) > 0.10:
#     flag()

# Example:

# AI model release

# Yesterday:
# 35%

# Today:
# 52%

# Move:
# +17%
# Signal 2: Liquidity opportunities
# if spread > 0.08:
#     flag()
# Signal 3: Volume spikes
# if volume_today > 5 * average_volume:
#     flag()

# Your output:

# TOP MARKETS TO REVIEW

# 1.
# Question:
# Will Fed cut rates?

# Price:
# 42%

# 24h move:
# +12%

# Reason:
# Large movement


# 2.
# Question:
# Will Company X acquire Y?

# Price:
# 33%

# Spread:
# 11 cents

# Reason:
# Wide market
# Day 5 — Add your probability workflow

# Do NOT automate this yet.

# Create a manual table:

# trade_journal.csv

# market,current_price,my_probability,edge,reason
# Fed cut,0.42,0.55,0.13,"Inflation falling"
# AI launch,0.35,0.45,0.10,"Company comments"

# The key question:

# Market probability:
# 42%

# My probability:
# 55%

# Difference:
# +13%
# Day 6 — Paper trading

# Before your C++ engine touches anything:

# Create:

# paper_buy(
#     market,
#     price,
#     size
# )

# Track:

# Position:
# YES Fed cut

# Entry:
# 42c

# Size:
# $100

# Current:
# 48c

# P&L:
# +$14
# Day 7 — Analytics

# Calculate:

# Return
# profit / capital
# Drawdown

# Largest loss from peak.

# Sortino

# Track:

# returns
# negative returns only

# Your output:

# Paper Portfolio

# Trades:
# 18

# Win rate:
# 61%

# Return:
# +8.4%

# Max drawdown:
# -2.1%

# Sortino:
# 2.4